## Transform Sprints Data
1. Read bronze sprints table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constructorId → constructor_id, driverId → driver_id, raceName → race_name, positionText → finish_position_text)
4. Rename columns to make them more meaningful (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position)
5. Filter out rows where season, round, custructor_id or driver_id is null (business key validation)
6. Remove duplicate records
7. Transform values of column race_name to Title Case
8. Write the transformed data to silver sprints table1. 

In [0]:
# 1. Get batch ID
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/03_silver_helpers"

In [0]:
from pyspark.sql import functions as f

In [0]:
bronze_path = "s3://bizf1project123/Bronze/Kalashri_increment/sprints"

silver_path = "s3://bizf1project123/Silver/Kalashri_increment/sprints"

## Step 1 to 4 Read Source Data, Select required columns & Standardise column names

In [0]:
sprints_df = (
    spark.read.format("delta")
        .load(bronze_path)
        .filter(F.col("batch_id") == v_batch_id.strip())
        .select(
            "season", "round", "constructorId", "driverId",
            "date", "raceName", "grid", "laps", "number",
            "points", "position", "positionText", "status",
            "ingestion_timestamp", "source_file", "batch_id"
        )
        .withColumnsRenamed({
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "raceName": "race_name",
            "date": "race_date",
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "position": "final_position",
            "positionText": "final_position_text"
        })
)

display(sprints_df)

## Step 5 & 6 Data Quality Checks
-    Filter out rows where season, round, custructor_id or driver_id is null (business key validation)
-    Remove duplicate records

In [0]:
sprints_valid_df = (
    sprints_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull() 
        )
        .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
)

display(sprints_valid_df)

In [0]:
display(sprints_df.count() - sprints_valid_df.count())

## Step 7 - Transform values of column nationality to Title Case

In [0]:
sprints_final_df = (
    sprints_valid_df
        .withColumn('race_name', F.initcap(F.col("race_name")))
)

display(sprints_final_df)

## Step 8 - Write the transformed data to silver sprints table

In [0]:
write_to_silver(
    input_df=sprints_final_df,
    target_path=silver_path,
    merge_condition="t.season = s.season AND t.round = s.round AND t.constructor_id = s.constructor_id AND t.driver_id = s.driver_id",
    columns_to_update=[
        "race_name",
        "race_date",
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "final_position",
        "final_position_text",
        "status",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
%sql
SELECT *
FROM delta.`s3://bizf1project123/Silver/Kalashri_increment/sprints`;

In [0]:
# dbutils.fs.rm(
#     "s3://bizf1project123/Silver/Kalashri_increment/sprints",
#     True
# )